In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import h5py, os, tqdm, glob, scipy
# os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.49'
import numpy as np
import matplotlib.pyplot as plt
from functools import partial

import jax
import jax.numpy as jnp
import jax_cosmo as jc

import optax
from optax.losses import huber_loss
from flax import nnx
import orbax.checkpoint as ocp
import jraph

import diffrax
from diffrax import diffeqsolve, ODETerm, LeapfrogMidpoint, PIDController, SaveAt, ConstantStepSize

import jaxpm
from jaxpm.painting import cic_paint, cic_read, compensate_cic
from jaxpm.kernels import fftk, gradient_kernel, invlaplace_kernel, longrange_kernel, invnabla_kernel
from jaxpm.utils import power_spectrum, cross_correlation_coefficients
from jaxpm.nn import MLP, ScaleConditionedCNN
from jaxpm import camels, plotting, hpm, nn, graph, diagnostics

# print(jax.devices("gpu"))
print(jax.default_backend())

/global/common/software/des/athomsen/flatiron/lib/python3.11/site-packages/jax_cosmo/__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound


gpu


# configuration

In [3]:
parts_per_dim = 64
# mesh_per_dim = parts_per_dim
mesh_per_dim = 2 * parts_per_dim

# parts_per_dim = 128
# mesh_per_dim = parts_per_dim

# parts_per_dim = None
# mesh_per_dim = 256

mesh_shape = [mesh_per_dim] * 3
box_size = [float(mesh_per_dim)] * 3

# i_snapshots = None
# i_snapshots = range(1, 33+8, 8)
# i_snapshots = range(1, 33+4, 4)
# i_snapshots = np.arange(-4, 0, dtype=int)
# i_snapshots = np.arange(-8, 0, dtype=int)

# CAMELS

In [4]:
CODE = "SIMBA"
# CODE = "ASTRID"
# CODE = "IllustrisTNG"

train_dict = camels.load_CV_snapshots(
    "CV_0",
    mesh_per_dim,
    parts_per_dim,
    i_snapshots=None,
    CAMELS="/pscratch/sd/a/athomsen/flatiron/CAMELS",
    CODE=CODE,
)

vali_dict = camels.load_CV_snapshots(
    "CV_1",
    mesh_per_dim,
    parts_per_dim,
    i_snapshots=None,
    CAMELS="/pscratch/sd/a/athomsen/flatiron/CAMELS",
    CODE=CODE,
)

Loaded /pscratch/sd/a/athomsen/flatiron/CAMELS/h5/SIMBA/CV/CV_0/parts=64,mesh=128.h5
Loaded /pscratch/sd/a/athomsen/flatiron/CAMELS/h5/SIMBA/CV/CV_1/parts=64,mesh=128.h5


In [5]:
cosmo = train_dict["cosmo"]
scales = train_dict["scales"]

dm_poss = train_dict["dm_poss"]
dm_vels = train_dict["dm_vels"]

gas_poss = train_dict["gas_poss"]
gas_vels = train_dict["gas_vels"]

In [6]:
# dt0 = 0.01
dt0 = 0.01

print(np.diff(scales))
assert np.all(np.diff(scales) > dt0)

[0.02380952 0.03333333 0.02222222 0.02777778 0.01224106 0.01284044
 0.01346916 0.01412867 0.01482047 0.01554614 0.01630734 0.01710582
 0.0179434  0.01882198 0.01974359 0.02071031 0.02172438 0.0227881
 0.0239039  0.02507434 0.02630208 0.02758995 0.02894087 0.03035793
 0.03184439 0.03340362 0.03503921 0.03675488 0.03855455 0.04044235
 0.04242258 0.04449977 0.04667867]


In [7]:
vcic_paint = jax.vmap(cic_paint, in_axes=(None,0,None))
vcic_read = jax.vmap(cic_read, in_axes=(0,0))

In [8]:
def train_step_base(model, optimizer, loss_fn_wrapper):
    loss, grads = nnx.value_and_grad(loss_fn_wrapper)(model)
    optimizer.update(grads)

    squared_sum = jax.tree_util.tree_reduce(
            lambda x, y: x + jnp.sum(y**2),
            grads,
            0.0
        )
    grad_norm = jnp.sqrt(squared_sum)

    return loss, grad_norm


@nnx.jit(static_argnames=("architecture",))
# def train_step_particle(model, optimizer, y0, ts, ref_pos, ref_cls, architecture="mlp"):
def train_step_particle(model, optimizer, y0, ts, ref_pos, architecture="mlp"):
    """dynamic snapshot range as passed"""
    
    def loss_fn_wrapper(model):
        # return particle_loss_fn(model, y0, ts, ref_pos, ref_cls, architecture, eps=1e-8)
        return particle_loss_fn(model, y0, ts, ref_pos, architecture, eps=1e-8)

    return train_step_base(model, optimizer, loss_fn_wrapper)
    
@nnx.jit(static_argnames=("architecture",))
def train_step_field(model, optimizer, y0, ts, ref_rho, ref_cls, architecture="cnn"):
    """dynamic snapshot range as passed"""
    
    def loss_fn_wrapper(model):
        return field_loss_fn(model, y0, ts, ref_rho, ref_cls, architecture, eps=1e-8)

    return train_step_base(model, optimizer, loss_fn_wrapper)

In [9]:
# edges = graph.get_edges(gas_poss, scales, k=4)

def solve_ode(model, y0, ts, architecture, training=True):
    ode = ODETerm(
        hpm.get_hpm_network_ode_fn(
            mesh_per_dim, 
            cosmo, 
            gravity_model=None, 
            pressure_model=model, 
            gas_architecture=architecture, 
            # precomputed_edges=edges,
            training=training,
        )
    )

    res = diffeqsolve(
            terms=ode,
            solver=LeapfrogMidpoint(),
            t0=ts[0],
            t1=ts[-1],
            dt0=dt0,
            y0=y0,
            saveat=SaveAt(ts=ts),
            max_steps=100,
            stepsize_controller=ConstantStepSize(),
        )
    res = res.ys

    return res


# loss

### CAMELS ground truth

In [10]:
# per-particle reference
ref_pos = jnp.stack(gas_poss, axis=0)
ref_vel = jnp.stack(gas_vels, axis=0)

# # velocity
# vel_mean_ref = jax.vmap(
#     jax.vmap(
#         cic_paint, 
#         in_axes=(None,0,0)
#     ),
#     in_axes=(None,None,-1), out_axes=-1,
# )(jnp.zeros(mesh_shape), ref_pos, ref_vel / ref_N[..., jnp.newaxis])

# ref_vel_mean = jax.vmap(
#     jax.vmap(
#         cic_read, 
#         in_axes=(0,0)
#     ),
#     in_axes=(-1,None), out_axes=-1,
# )(vel_mean_ref, ref_pos)

# ref_vel_disp = jnp.sum((ref_vel_mean - ref_vel) ** 2, axis=-1)

# # field-level reference
# gas_mass = cosmo.Omega_b / (cosmo.Omega_b + cosmo.Omega_c)
# ref_rho = vcic_paint(jnp.zeros(mesh_shape), gas_poss, gas_mass)

# # power spectrum reference
# vpower_spectrum = jax.vmap(
#     lambda fields: 
#         power_spectrum(
#             compensate_cic(fields),
#             boxsize=np.array([25.0] * 3),
#             kmin=np.pi / 25.0,
#             dk=2 * np.pi / 25.0,
#         )
# )
# _, ref_cls = vpower_spectrum(ref_rho)

# vcross_correlation_separate = jax.vmap(
#     lambda field_a, field_b:
#         cross_correlation_coefficients(
#             compensate_cic(field_a),
#             compensate_cic(field_b),
#             boxsize=np.array([25.0] * 3),
#             kmin=np.pi / 25.0,
#             dk=2 * np.pi / 25.0,
#         )
# )
# vcross_correlation = lambda rhos: vcross_correlation_separate(rhos, ref_rho)

### particle-level

In [11]:
# def particle_loss_fn(model, y0, ts, ref_pos, ref_cls, architecture, eps=1e-8):
def particle_loss_fn(model, y0, ts, ref_pos, architecture, eps=1e-8):
    print("using particle loss")
    
    res = solve_ode(model, y0, ts, architecture)
    gas_poss = res[2]
    gas_vels = res[3]

    delta_pos = ((gas_poss - ref_pos + mesh_per_dim // 2) % mesh_per_dim) - mesh_per_dim // 2
    pos_loss = jnp.sum(delta_pos**2, axis=-1)
    # pos_loss /= jnp.maximum(scales[:, jnp.newaxis]**3, eps)
    pos_loss = jnp.mean(pos_loss)

    # res_rho = vcic_paint(jnp.zeros(mesh_shape), gas_poss, gas_mass)
    # kbins, res_cls = vpower_spectrum(res_rho)

    # k = kbins[0]
    # k_min, k_cutoff = k[0], k[int(0.2*len(k))]
    # k_weights = jnp.expand_dims(jnp.exp(-(k - k_min) / k_cutoff), 0)

    # cl_loss = jnp.sum(((res_cls/jnp.maximum(ref_cls, eps) - 1)*k_weights)**2, axis=-1)
    # # cl_loss /= jnp.maximum(scales[:,jnp.newaxis]**3, eps)
    # cl_loss = jnp.mean(cl_loss)

    # return pos_loss + 0.1 * cl_loss

    return pos_loss


def batched_particle_loss_fn(model, y0_batch, ts_batch, ref_batch, architecture, eps=1e-8):
    """Process multiple initial conditions in a batch"""
    def single_loss_fn(model, y0, ts, ref):
        return particle_loss_fn(model, y0, ts, ref, architecture, eps=1e-8)
        
    batch_losses = jax.vmap(single_loss_fn, in_axes=(None,0,0,0))(model, y0_batch, ts_batch, ref_batch)
    
    return jnp.mean(batch_losses)

In [12]:
# def particle_loss_fn(model, architecture):
#     res = solve_ode_full(model, architecture)
#     gas_poss = res[2]%mesh_per_dim
#     gas_vels = res[3]

#     pos_loss = jnp.sum((gas_poss - ref_pos)**2, axis=-1)
#     pos_loss = jnp.where(pos_loss < mesh_per_dim//2, pos_loss, 0.0)
#     # pos_loss *= jnp.expand_dims(scales, axis=1)
#     pos_loss = jnp.mean(pos_loss)

#     vel_loss = jnp.sum((gas_vels - ref_vel)**2, axis=-1)
#     vel_loss = jnp.where(vel_loss < mesh_per_dim//2, vel_loss, 0.0)
#     # vel_loss *= jnp.expand_dims(scales, axis=1)
#     vel_loss = jnp.mean(vel_loss)

#     res_rho = vcic_paint(jnp.zeros(mesh_shape), gas_poss, gas_mass)
#     _, res_cls = vpower_spectrum(res_rho)
#     cl_loss = jnp.mean(jnp.sum((res_cls/ref_cls - 1)**2, axis=-1))

#     # _, res_cross = vcross_correlation(res_rho)
#     # cross_loss = jnp.mean(jnp.sum((res_cross/jnp.sqrt(ref_cls * res_cls) - 1)**2, axis=-1))

#     return pos_loss + 0.01 * vel_loss + 0.1 * cl_loss


### field-level

In [13]:
# def field_loss_fn(model, y0, ts, ref_rho, ref_cls, architecture, eps=1e-8):
#     print("using field loss")
    
#     res = solve_ode(model, y0, ts, architecture)
#     gas_poss = res[2]%mesh_per_dim
#     gas_vels = res[3]

#     res_rho = vcic_paint(jnp.zeros(mesh_shape), gas_poss, gas_mass)
#     rho_loss = (res_rho - ref_rho)**2
#     # rho_loss /= jnp.maximum(scales.reshape(-1,1,1,1)**2, eps)
#     rho_loss = jnp.mean(rho_loss)

#     # kbins, res_cls = vpower_spectrum(res_rho)
#     # k = kbins[0]
#     # k_min, k_cutoff = k[0], k[int(0.2*len(k))]
#     # k_weights = jnp.expand_dims(jnp.exp(-(k - k_min) / k_cutoff), 0)

#     # cl_loss = jnp.sum(((res_cls/jnp.maximum(ref_cls, eps) - 1)*k_weights)**2, axis=-1)
#     # # cl_loss /= jnp.maximum(scales[:,jnp.newaxis]**3, eps)
#     # cl_loss = jnp.mean(cl_loss)
        
#     # return rho_loss + 0.1 * cl_loss
#     return rho_loss


In [14]:
# def _huber(x, delta=0.5):
#     ax = jnp.abs(x)
#     return jnp.where(ax <= delta, 0.5 * x**2, delta * (ax - 0.5 * delta))


# def _delta(field, eps=1e-8):
#     mean = jnp.mean(field, axis=(-3, -2, -1), keepdims=True)
#     return field / (mean + eps) - 1.0


# def _asinh_stabilize(delta, ref_delta, eps=1e-8):
#     # per-sample sigma from reference stabilizes scale
#     sigma = jnp.std(ref_delta, axis=(-3, -2, -1), keepdims=True) + eps
#     return jnp.arcsinh(delta / sigma)


# def field_loss_fn(
#     model,
#     y0,
#     ts,
#     ref_rho,
#     ref_cls,
#     architecture,
#     eps=1e-8,
#     w_real=1.0,
#     w_grad=0.5,
#     w_ps=0.1,
#     k_frac=0.2,
#     huber_delta=0.5,
# ):
#     res = solve_ode(model, y0, ts, architecture)
#     gas_poss = res[2] % mesh_per_dim
#     gas_vels = res[3]

#     # Paint predicted field
#     res_rho = vcic_paint(jnp.zeros(mesh_shape), gas_poss, gas_mass)

#     # Work with density contrast and stabilized transform
#     res_delta = _delta(res_rho, eps)
#     ref_delta = _delta(ref_rho, eps)

#     res_t = _asinh_stabilize(res_delta, ref_delta, eps)
#     ref_t = _asinh_stabilize(ref_delta, ref_delta, eps)

#     # Robust real-space loss on stabilized field
#     real_loss = jnp.mean(_huber(res_t - ref_t, delta=huber_delta))

#     return real_loss


# architecture

### latent

In [15]:
# with_latent = True
with_latent = False

if with_latent:
    # latent_init = cic_read(cic_paint(jnp.zeros(mesh_shape), gas_poss[0]), gas_poss[0])
    # latent_init = jnp.expand_dims(latent_init, axis=-1)
    
    latent_init = jnp.ones((parts_per_dim**3,1))
    # latent_init = jnp.ones((parts_per_dim**3,8))
    # latent_init = np.random.normal(loc=1.0, scale=0.01, size=(parts_per_dim**3,4))
    
    # for field cnn
    # latent_init = np.random.normal(loc=1.0, scale=0.01, size=tuple(mesh_shape) + (4,))

    latent_dim = latent_init.shape[-1]
else:
    latent_dim = 0

### MLP

In [16]:
model = MLP(
    d_in=5 + latent_dim,
    d_out=1 + latent_dim, 
    # d_hidden=16, 
    # d_hidden=64, 
    d_hidden=128,
    # d_hidden=256,
    n_hidden=4, 
    # n_hidden=2, 
    # dropout_rate=0.01,
    dropout_rate=0.0,
    rngs=nnx.Rngs(0),
    norm_type="layer",
    # norm_type="batch",
    # activation=jax.nn.relu,
    # activation=jax.nn.tanh,
    activation=jax.nn.swish,
)
architecture = "mlp"

### CNN

In [17]:
# model = ScaleConditionedCNN(
#     d_in=4 + latent_dim, 
#     d_out=1 + latent_dim,
#     d_hidden=64,
#     # d_hidden=128,
#     # d_hidden=256,
#     # n_hidden=4,
#     n_hidden=4,
#     kernel_size=(3, 3, 3),
#     rngs=nnx.Rngs(0),
#     norm_type="layer",
#     activation=jax.nn.swish,
# )

# architecture = "cnn"

### MLP + CNN

In [18]:
# mlp = MLP(
#     d_in=5 + latent_dim,
#     d_out=8, 
#     d_hidden=64, 
#     n_hidden=4, 
#     rngs=nnx.Rngs(0)
# )

# cnn = CNN(
#     d_in=4 + latent_dim, 
#     d_out=8,
#     d_hidden=16,
#     n_hidden=1,
#     kernel_size=(3, 3, 3),
#     strides=1,
#     rngs=nnx.Rngs(0)
# )

# model = HybridNet(
#     mlp,
#     cnn,
#     d_out=1 + latent_dim, 
#     rngs=nnx.Rngs(0)
# )

# architecture = "mlp+cnn"

In [19]:
# if latent_init is None:
#     latent_dim = 0
# else:
#     latent_dim = latent_init.shape[-1]

# mlp = MLP(
#     d_in=5 + latent_dim,
#     d_out=16, 
#     d_hidden=64, 
#     n_hidden=4, 
#     rngs=nnx.Rngs(0)
# )

# cnn = CNN(
#     d_in=4 + latent_dim, 
#     d_out=16,
#     d_hidden=16,
#     n_hidden=1,
#     kernel_size=(5, 5, 5),
#     strides=1,
#     rngs=nnx.Rngs(0)
# )

# model = HybridNet(
#     mlp,
#     cnn,
#     d_out=1 + latent_dim, 
#     rngs=nnx.Rngs(0)
# )

# architecture = "mlp+cnn"

### GNN

on the fly

In [20]:
# with_latent = False

# model = AttentionGNN(
#     d_node=5 + with_latent,
#     d_edge=1,
#     d_query=32,
#     n_hidden=4,
#     d_out=1 + with_latent,
#     rngs=nnx.Rngs(0),
# )

# architecture = "gnn"

# training

In [21]:
def train_step_wrapper(i_step=None):
    if i_step is None:
        i_step = np.arange(0, len(scales))

    i0, i1 = i_step[0], i_step[-1]
    y0 = (dm_poss[i0], dm_vels[i0], gas_poss[i0], gas_vels[i0])
    if with_latent:
        y0 += (latent_init,)
    ts = scales[i_step]

    if architecture in ["mlp"]:
        loss, grad_norm = train_step_particle(
            model, 
            optimizer, 
            y0, 
            scales[i_step], 
            ref_pos[i_step], 
            # ref_cls[i_step],
        )
    elif architecture in ["cnn"]:
        loss, grad_norm = train_step_field(
            model, 
            optimizer, 
            y0, 
            scales[i_step], 
            ref_rho[i_step],
            ref_cls[i_step],
        )
    else:
        raise NotImplementedError

    losses.append(float(loss))
    grad_norms.append(float(grad_norm))
    pbar.set_description(f"Loss: {loss:.4e}, Grad norm: {grad_norm:.4e}, {i_step}")

def plot_training(losses, grad_norms):
    fig, ax = plt.subplots(figsize=(6,10), nrows=2, sharex=True)
    ax[0].plot(losses)
    ax[0].set(yscale="log", title="loss")
    
    ax[1].plot(grad_norms)
    ax[1].set(yscale="log", title="norm(grad)")

In [22]:
total_steps = 100

# learning_rate = 1e-3
# learning_rate = 1e-4
# learning_rate = 1e-5
# learning_rate = optax.cosine_decay_schedule(
#     init_value=1e-4, 
#     decay_steps=total_steps, 
#     alpha=0.1
# )
learning_rate = optax.warmup_cosine_decay_schedule(
    init_value=1e-5,
    peak_value=1e-4,
    end_value=1e-5,
    warmup_steps=total_steps//5,
    decay_steps=total_steps - total_steps//5, 
)
clip_norm = 1

optimizer = nnx.ModelAndOptimizer(
    model,
    optax.chain(
        optax.clip_by_global_norm(clip_norm),
        optax.adam(learning_rate)
        # optax.ema(decay=0.999),
        # optax.adamw(learning_rate, b1=0.95, b2= 0.9999, eps=1e-5)
        # optax.adamw(learning_rate, eps=1e-5)
    )
)

losses = []
grad_norms = []

In [23]:
# i_snapshots = np.arange(-4, 0, dtype=int)
# i_snapshots = np.arange(0, 4, dtype=int)
# i_snapshots = np.arange(12, 16, dtype=int)

# i_snapshots = np.arange(-8, 0, dtype=int)
# i_snapshots = np.arange(0, 8, dtype=int)
# i_snapshots = np.arange(1, 33+8, 8)

## fixed snapshots

### all

In [24]:
for i in (pbar := tqdm.tqdm(range(total_steps))):
    train_step_wrapper()
plot_training(losses, grad_norms)

  0%|          | 0/300 [00:00<?, ?it/s]

using particle loss
dark matter and gas
Using learned pressure force
No latent variable
dark matter and gas
Using learned pressure force
No latent variable
dark matter and gas
Using learned pressure force
No latent variable


Loss: 2.2785e+01, Grad norm: 1.2139e+01, [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
Loss: 2.2785e+01, Grad norm: 1.2139e+01, [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
Loss: 2.2771e+01, Grad norm: 1.1328e+01, [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
Loss: 2.2771e+01, Grad norm: 1.1328e+01, [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
Loss: 2.2756e+01, Grad norm: 1.0448e+01, [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
Loss: 2.2756e+01, Grad norm: 1.0448e+01, [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
Loss: 2.2740e+01, Grad norm: 9.5106e+00, [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
Loss: 2.2740e+01, Grad norm: 9.5106e+00, [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
Loss: 2.2725e+01, Grad norm: 8.5501e+00, [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 1

KeyboardInterrupt: 

### subset

In [ ]:
# for i in (pbar := tqdm.tqdm(range(total_steps))):
#     train_step_wrapper(i_snapshots)
# plot_training(losses, grad_norms)

## random snapshots

### spaced

In [ ]:
# i_base = np.arange(0, 32, 4)
# i_delta = 6

# for i in (pbar := tqdm.tqdm(range(total_steps))):
#     i_shift = np.random.randint(0, i_delta)
#     i_step = i_base + i_shift

#     train_step_wrapper(i_step)
# plot_training(losses, grad_norms)

### continuous

In [ ]:
# i_delta = 8

# for i in (pbar := tqdm.tqdm(range(total_steps))):
#     i_rand = np.random.randint(0, len(scales) - i_delta)
#     i_step = np.arange(i_rand, i_rand + i_delta)
    
#     train_step_wrapper(i_step)
# plot_training(losses, grad_norms)

### checkpointing

In [ ]:
# # see https://flax.readthedocs.io/en/latest/guides/checkpointing.html
# # checkpoint_file = os.path.join(os.getcwd(), "checkpoints/mlp_sim_v1_late_time.jx")
# # checkpoint_file = os.path.join(os.getcwd(), "checkpoints/mlp_sim_v2.jx")
# # checkpoint_file = os.path.join(os.getcwd(), "checkpoints/mlp_sim_illustris_v1.jx")
# checkpoint_file = os.path.join(os.getcwd(), "checkpoints/mlp_sim_v4_full.jx")
# checkpointer = ocp.StandardCheckpointer()
# print(os.getcwd())

In [ ]:
# _, params = nnx.split(model)
# checkpointer.save(checkpoint_file, params, force=True)

In [ ]:
# abstract_model = nnx.eval_shape(lambda: model)
# graphdef, abstract_params = nnx.split(abstract_model)

# params = checkpointer.restore(checkpoint_file, abstract_params)
# model = nnx.merge(graphdef, params)

# run the simulation

In [ ]:
for camels_dict in [train_dict, vali_dict]:
# for camels_dict in [train_dict]:
    diagnostics.run_simulations(
        camels_dict,
        mesh_per_dim,
        pressure_model=model, 
        gas_architecture=architecture, 
        dt0=dt0,
        # i_snapshots=None,
        # i_snapshots=i_snapshots,
        i_snapshots=range(0, 32+8, 8),
        # i_snapshots = range(1, 33+8, 8),
        # i_snapshots = range(1, 33+4, 4),
        plot_dm=False,
        plot_gas=True,
        plot_latent=with_latent,
    )